# 02. Sinüzoidal Pozisyonel Dalga Frekansları ve Mesafe Analizi

Bu notebook, "Attention Is All You Need" (Vaswani et al., 2017) Bölüm 3.5'te önerilen
analitik sinüzoidal pozisyonel kodlama fonksiyonunun geometrik yapısını ve dalga frekanslarını görselleştirir.

Formüller:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
import sys
sys.path.append("..")
import torch
import matplotlib.pyplot as plt
import numpy as np
from src import PositionalEncoding

# Model Parametreleri
d_model = 128
max_len = 100

pe_layer = PositionalEncoding(d_model=d_model, dropout=0.0, max_len=max_len)
# [1, max_len, d_model] -> [max_len, d_model]
pe_matrix = pe_layer.pe.squeeze(0).cpu().numpy()

print(f"Pozisyonel Kodlama Matrisi Şekli: {pe_matrix.shape}")

## 1. Pozisyonel Kodlama Isı Haritası (Heatmap)
Düşük boyut indeksleri (sol) yüksek frekansta hızla salınırken, yüksek boyut indeksleri (sağ) geniş dalga boylarına sahiptir.

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(pe_matrix, cmap='viridis', aspect='auto')
plt.title('Sinüzoidal Pozisyonel Kodlama Isı Haritası (PE Matrix)', fontsize=14, pad=12)
plt.xlabel('Gizli Boyut İndeksi (i)', fontsize=12)
plt.ylabel('Pozisyon (pos)', fontsize=12)
plt.colorbar(label='Kodlama Değeri')
plt.tight_layout()
plt.show()

## 2. Farklı Boyutlardaki Frekans Salınımları
Boyut arttıkça frekansın nasıl azaldığını açıkça görebiliriz.

In [ ]:
plt.figure(figsize=(12, 5))
positions = np.arange(max_len)

plt.plot(positions, pe_matrix[:, 0], label='Boyut 0 (En Yüksek Frekans)', color='#e74c3c', lw=2)
plt.plot(positions, pe_matrix[:, 4], label='Boyut 4 (Orta-Yüksek Frekans)', color='#e67e22', lw=2)
plt.plot(positions, pe_matrix[:, 16], label='Boyut 16 (Orta Frekans)', color='#2ecc71', lw=2)
plt.plot(positions, pe_matrix[:, 64], label='Boyut 64 (Düşük Frekans)', color='#3498db', lw=2)

plt.title('Farklı Gizli Boyutlarda Dalga Salınımları', fontsize=14)
plt.xlabel('Pozisyon (pos)', fontsize=12)
plt.ylabel('Değer', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Pozisyonlar Arası Benzerlik ve Mesafe İlişkisi
Birbirine yakın pozisyonların iç çarpımı yüksek, uzak pozisyonlarınki düşüktür. Bu özellik modelin göreli mesafeleri öğrenmesini sağlar.

In [ ]:
# Nokta çarpım benzerlik matrisi: PE * PE^T
similarity = np.dot(pe_matrix, pe_matrix.T)

plt.figure(figsize=(8, 7))
plt.imshow(similarity, cmap='magma')
plt.title('Pozisyonlar Arası Nokta Çarpım Benzerliği (PE · PE^T)', fontsize=14, pad=12)
plt.xlabel('Pozisyon (pos_1)', fontsize=12)
plt.ylabel('Pozisyon (pos_2)', fontsize=12)
plt.colorbar(label='İç Çarpım Değeri')
plt.tight_layout()
plt.show()